In [91]:
import pandas as pd
import sqlite3

In [92]:
df = pd.read_csv('../data/superstore.csv', encoding='latin-1')
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date'] = pd.to_datetime(df['Ship Date'])
df['Order Year'] = df['Order Date'].dt.year
df['Order Month'] = df['Order Date'].dt.month

In [93]:
conn = sqlite3.connect(':memory:')
df.to_sql('superstore', conn, index=False, if_exists='replace')

def run_query(sql, currency_cols=None):
    result = pd.read_sql_query(sql, conn)
    if currency_cols:
        return result.style.format({col: '${:,.2f}' for col in currency_cols})
    return result

#### Q1: Total Profit and Sales by Region (Ranked)

In [94]:
run_query("""
          SELECT Region, ROUND(SUM(Profit), 2) AS total_profit, ROUND(SUM(Sales), 2) AS total_sales
          FROM superstore
          GROUP BY Region
          ORDER BY total_profit DESC
""", currency_cols=['total_profit', 'total_sales'])

,Region,total_profit,total_sales
0,West,"$108,418.45","$725,457.82"
1,East,"$91,522.78","$678,781.24"
2,South,"$46,749.43","$391,721.91"
3,Central,"$39,706.36","$501,239.89"


Most profit and sales come from West region, their profit is 15% of their sales. Central region has the least profit, but not the least sales. Their profit is 8% of their sales. South region has the least sales but not profit. Their profit is 12% of their sales, which makes it overall higher than Central region - meaning Central is less efficient at turning sales into profit.

#### Q2: Total Profit and Sales by Category

In [95]:
run_query("""
          SELECT Category, ROUND(SUM(Profit), 2) AS total_profit, ROUND(SUM(Sales), 2) AS total_sales
          FROM superstore
          GROUP BY Category
          ORDER BY total_profit DESC
""", currency_cols=['total_profit', 'total_sales'])

,Category,total_profit,total_sales
0,Technology,"$145,454.95","$836,154.03"
1,Office Supplies,"$122,490.80","$719,047.03"
2,Furniture,"$18,451.27","$741,999.80"


Most profit and sales come from Technology category, their profit is 17% of their sales. Furniture category has the least profit, but not the least sales. Their profit is 2% of their sales - makes it very low when compared to Technology. Office Supplies has the least sales but achieves the same 17% profit margin as Technology - meaning Furniture stands out as the only underperforming category, converting just 2% of its sales into profit.

#### Q3: Which Sub-Categories are Loss-Making?

In [96]:
run_query("""
          SELECT Category, `Sub-Category`, ROUND(SUM(Profit),2) AS total_loss_profit
          FROM superstore
          GROUP BY `Sub-Category`
          HAVING SUM(Profit) < 0
          ORDER BY total_loss_profit ASC
""", currency_cols=['total_loss_profit'])

,Category,Sub-Category,total_loss_profit
0,Furniture,Tables,"$-17,725.48"
1,Furniture,Bookcases,"$-3,472.56"
2,Office Supplies,Supplies,"$-1,189.10"


From Q2 we knew that Furniture is the least profitable category. Tables seem to be the least profitable sub-category within furniture with a loss of $17k. Bookcases generate a $3k loss from the Furniture category. Supplies generates a $1k loss but from the Office Supplies category. Even though the loss on Supplies is relatively small, it is worth investigating whether these losses are driven by heavy discounting or pricing issues before any action is taken.

#### Q4: Average Profit Margin by Category

In [97]:
run_query("""
          SELECT Category, ROUND((SUM(Profit) / SUM(Sales)) * 100, 2) as average_profit_margin
          FROM superstore
          GROUP BY Category
          ORDER BY average_profit_margin DESC
""")

,Category,average_profit_margin
0,Technology,17.40
1,Office Supplies,17.04
2,Furniture,2.49


Technology and Office Supplies both have a 17% profit margin, while Furniture only has 2%. This means that Furniture's issue is not just low sales - it is also making less money from those sales compared to the other categories.

#### Q5: How Many Orders Have Discounts Above 40%, and What Is Their Total Loss?

In [98]:
run_query("""
          SELECT COUNT(*) AS total_order_disc_abv_40, ROUND(SUM(Profit), 2) AS total_loss_profit
          FROM superstore
          WHERE Discount > 0.4 
""", currency_cols=['total_loss_profit'])

,total_order_disc_abv_40,total_loss_profit
0,933,"$-99,558.59"


There are 933 orders with discounts above 40% generating a combined loss of $99k. The average loss per order is around $106 - meaning every heavily discounted order costs the business money on average.

#### Q6: Sales and Profit by Customer Segment

In [99]:
run_query("""
          SELECT Segment, ROUND(SUM(Profit), 2) AS total_profit, ROUND(SUM(Sales), 2) AS total_sales
          FROM superstore
          GROUP BY Segment
          ORDER BY total_profit DESC
""", currency_cols=['total_profit', 'total_sales'])

,Segment,total_profit,total_sales
0,Consumer,"$134,119.21","$1,161,401.34"
1,Corporate,"$91,979.13","$706,146.37"
2,Home Office,"$60,298.68","$429,653.15"


Most profit and sales come from Consumer segment with 11% profit margin. Corporate segment has lower profit and sales than Consumer, but it generates higher profit margin at 13%. The lowest profit and sales come from Home Office, but their profit margin seems to be the highest out of segments at 14%. This is worth investigating further - if the business can grow Home Office sales volume, it may generate more profit per dollar than the other segments.

#### Q7: Top 10 Most Profitable Individual Products

In [100]:
run_query("""
          SELECT `Product Name`, Category, `Sub-Category`, ROUND(SUM(Profit), 2) AS total_profit
          FROM superstore
          Group BY `Product Name`
          ORDER BY total_profit DESC
          LIMIT 10
""", currency_cols=['total_profit'])

,Product Name,Category,Sub-Category,total_profit
0,Canon imageCLASS 2200 Advanced Copier,Technology,Copiers,"$25,199.93"
1,Fellowes PB500 Electric Punch Plastic Comb Binding Machine with Manual Bind,Office Supplies,Binders,"$7,753.04"
2,Hewlett Packard LaserJet 3310 Copier,Technology,Copiers,"$6,983.88"
3,Canon PC1060 Personal Laser Copier,Technology,Copiers,"$4,570.93"
4,"HP Designjet T520 Inkjet Large Format Printer - 24"" Color",Technology,Machines,"$4,094.98"
5,Ativa V4110MDD Micro-Cut Shredder,Technology,Machines,"$3,772.95"
6,"3D Systems Cube Printer, 2nd Generation, Magenta",Technology,Machines,"$3,717.97"
7,Plantronics Savi W720 Multi-Device Wireless Headset System,Technology,Accessories,"$3,696.28"
8,Ibico EPK-21 Electric Binding System,Office Supplies,Binders,"$3,345.28"
9,Zebra ZM400 Thermal Label Printer,Technology,Machines,"$3,343.54"


8 out of the top 10 most profitable products are from Technology, and 2 are from Office Supplies - consistent with Q2's finding that these are the two highest margin categories. Within Technology, 3 are Copiers, 4 are Machines and 1 Accessories, making them the leading sub-categories for profitability. The remaining 2 are Binders from Office Supplies. 

#### Q8: Bottom 10 Least Profitable Products

In [101]:
run_query("""
          SELECT `Product Name`, Category, `Sub-Category`, ROUND(SUM(Profit), 2) AS total_profit
          FROM superstore
          Group BY `Product Name`
          ORDER BY total_profit ASC
          LIMIT 10
""", currency_cols=['total_profit'])

,Product Name,Category,Sub-Category,total_profit
0,Cubify CubeX 3D Printer Double Head Print,Technology,Machines,"$-8,879.97"
1,Lexmark MX611dhe Monochrome Laser Printer,Technology,Machines,"$-4,589.97"
2,Cubify CubeX 3D Printer Triple Head Print,Technology,Machines,"$-3,839.99"
3,Chromcraft Bull-Nose Wood Oval Conference Tables & Bases,Furniture,Tables,"$-2,876.12"
4,Bush Advantage Collection Racetrack Conference Table,Furniture,Tables,"$-1,934.40"
5,GBC DocuBind P400 Electric Binding System,Office Supplies,Binders,"$-1,878.17"
6,Cisco TelePresence System EX90 Videoconferencing Unit,Technology,Machines,"$-1,811.08"
7,Martin Yale Chadless Opener Electric Letter Opener,Office Supplies,Supplies,"$-1,299.18"
8,Balt Solid Wood Round Tables,Furniture,Tables,"$-1,201.06"
9,BoxOffice By Design Rectangular and Half-Moon Meeting Room Tables,Furniture,Tables,"$-1,148.44"


4 out of the bottom 10 least profitable products are from Technology, 4 are from Furniture, and 2 are from Office Supplies. From Technology, all of them are Machines. From Furniture, all of them are Tables. And from Office Supplies, one of them is Binders and one is Supplies. This aligns with Q3, which 4 out of 10 items are Tables and one out of 10 is Supplies which is the least profitable sub-category. These products are actively costing the business money - combined loss of $29,458.38. Especially, Machines appear in both the top 10 most profitable (Q7) and the bottom 10 least profitable products - suggesting high variance within the sub-category. Some Machine sell at full price for strong profit, while others are heavily discounted into losses. This is a pricing strategy problem worth investigating further.

#### Q9: Year-On-Year Sales and Profit Totals

In [104]:
run_query("""
          SELECT `Order Year`, ROUND(SUM(Profit), 2) AS total_profit, ROUND(SUM(Sales), 2) AS total_sales
          FROM superstore
          Group BY `Order Year`
          ORDER BY `Order Year` ASC
""", currency_cols=['total_profit', 'total_sales'])

,Order Year,total_profit,total_sales
0,2014,"$49,543.97","$484,247.50"
1,2015,"$61,618.60","$470,532.51"
2,2016,"$81,795.17","$609,205.60"
3,2017,"$93,439.27","$733,215.26"


Sales and profit both grew from 2014 to 2017, with 2017 generating the highest sales at $733k and the highest profit at $93k. The profit margin improved from 10% in 2014 to 13% in 2017, meaning the business is becoming more efficient at turning sales into profit over time. Interestingly, 2015, 2016, and 2017 all share a similar 13% margin despite having very different sales volumes - this is worth investigating further to understand what drove the margin improvement between 2014 and 2015.

#### Q10: Which Region + Category Combination is the Most Loss-Making?

In [105]:
run_query("""
          SELECT Region, Category, COUNT(*) AS total_order, ROUND(SUM(Profit), 2) AS total_loss_profit
          FROM superstore
          Group BY Region, Category
          ORDER BY total_loss_profit DESC
""", currency_cols=['total_loss_profit'])

,Region,Category,total_order,total_loss_profit
0,West,Office Supplies,1897,"$52,609.85"
1,East,Technology,535,"$47,462.04"
2,West,Technology,599,"$44,303.65"
3,East,Office Supplies,1712,"$41,014.58"
4,Central,Technology,420,"$33,697.43"
5,South,Technology,293,"$19,991.83"
6,South,Office Supplies,995,"$19,986.39"
7,West,Furniture,707,"$11,504.95"
8,Central,Office Supplies,1422,"$8,879.98"
9,South,Furniture,332,"$6,771.21"


In [103]:
run_query("""
          SELECT Region, Category, COUNT(*) AS total_order, ROUND(SUM(Profit), 2) AS total_loss_profit
          FROM superstore
          Group BY Region, Category
          HAVING SUM(Profit) < 0
          ORDER BY total_loss_profit DESC
""", currency_cols=['total_loss_profit'])

,Region,Category,total_order,total_loss_profit
0,Central,Furniture,481,"$-2,871.05"


The only loss-making region and category combination is Furniture in the Central region, generating a $2,871 loss from 481 orders - an average loss of less than $6 per order. The loss is relatively small, suggesting it may be driven by a small number of heavily discounted orders rather than a fundamental problem with the category. This warrants further investigation in the next phase.